# **IMPORT**

In [1]:
import requests

from os import (
    environ,
    listdir )
from typing import Any
from joblib import load as joblib_load

import pandas as pd
import psycopg2

from pprint import pprint
from sklearn.pipeline import Pipeline

# **DATA**

In [2]:
dict_sample: dict = {
    "gender": "Female",
    "age": 0,
    "city": "string",
    "cgpa": 10,
    "degree": "M.Pharm",
    "profession": "Architect",
    "profession_hour_daily": 24,
    "profession_pressure": 0,
    "profession_satisfaction": 5,
    "financial_stress": 5,
    "sleep_duration": "Less than 5 hours",
    "dietary_habits": "Healthy",
    "history_family_illness_mental": True,
    "history_thoughts_suicidal": True }


dict_prediction: dict = {
    "date": "2026/08/25",
    "gender": "Female",
    "age": 40,
    "city": "Vault 777",
    "cgpa": 10,
    "degree": "M.Pharm",
    "profession": "Architect",
    "profession_hour_daily": 24,
    "profession_pressure": 5,
    "profession_satisfaction": 5,
    "financial_stress": 5,
    "sleep_duration": "Less than 5 hours",
    "dietary_habits": "Healthy",
    "history_family_illness_mental": True,
    "history_thoughts_suicidal": True,
    "depression": True,
    "confidence": 0.777 }

In [3]:
DIR_MODEL_SELECTED: str = environ.get("LOCAL_DIR_MODEL_SELECTED")

API_POST_PREDICT = environ.get("AI_API_POST_PREDICT") # "/predict", in ".env"

DB_PORT = environ.get("DB_PORT")
DB_NAME = environ.get("DB_NAME")
DB_USERNAME = environ.get("DB_USERNAME")
DB_PASSWORD = environ.get("DB_PASSWORD") 

In [7]:
def load_selected_model(dir_model: str,
                        model_ext: str = "pkl") -> Any:
    list_file_model: list[str] = []
    for file in listdir(path = dir_model):
        if (file.split(".")[-1] == model_ext):
            list_file_model.append(file)

    model_loaded: Any = None
    if (len(list_file_model) == 1):
        model: str = f"{dir_model}/{list_file_model[0]}"
        model_loaded = joblib_load(model)

    return model_loaded

# **Usage of the model**

In [8]:
test_input: pd.DataFrame = pd.DataFrame(
    data = [list(dict(dict_sample).values())],
    columns = list(dict(dict_sample).keys()) )

display(test_input)

test_model: Pipeline = load_selected_model(dir_model = DIR_MODEL_SELECTED)
test_model.predict(test_input)

,gender,age,city,cgpa,degree,profession,profession_hour_daily,profession_pressure,profession_satisfaction,financial_stress,sleep_duration,dietary_habits,history_family_illness_mental,history_thoughts_suicidal
0,Female,0,string,10,M.Pharm,Architect,24,0,5,5,Less than 5 hours,Healthy,True,True


array([ True])

## **Usage of the API (*/predict*)**

In [9]:
response: requests.Response = requests.post(
    url = f"http://localhost:8000{API_POST_PREDICT}",
    json = dict_sample )
pprint(dict(response.json()))

KeyboardInterrupt: 

## **Usage of the database (input data)**

In [11]:
def dict_keys_into_tuple_table_keys_str(dict_insert: dict) -> str:
    dict_tuple_keys: list[str] = tuple(dict_insert.keys())
    dict_tuple_keys_str: str = f"({", ".join(dict_tuple_keys)})"
    table_keys_str: str = dict_tuple_keys_str

    # print(table_keys_str)

    return table_keys_str


def list_placeholder_str_from_dict_length(dict_insert: str) -> str:
    count_placeholder: int = len(dict_insert)
    list_placeholder: list[str] = ["%s" for _ in range(count_placeholder)]
    list_placeholder_str: str = f"({", ".join(list_placeholder)})"

    # print(list_placeholder_str)

    return list_placeholder_str



# Connect to the database
conn = psycopg2.connect(
    host = "localhost",
    port = DB_PORT,
    database = DB_NAME,
    user = DB_USERNAME,
    password = DB_PASSWORD )

# Insert the data into the database
cur = conn.cursor()
cur.execute(query = f"""
                INSERT INTO prediction
                    {dict_keys_into_tuple_table_keys_str(dict_prediction)}
                VALUES
                    {list_placeholder_str_from_dict_length(dict_prediction)}
            """,
            vars = list(dict_prediction.values()) )
conn.commit()